In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Import threading, time, uuid
# 2. Define class Blackboard:
#    a. __init__: self._store = {}, self._lock = threading.Lock(), self.event_log = []
#    b. write(key, value, agent_id="system"): acquire lock; store entry with version,
#       agent_id, timestamp; append {"op":"write", "key", "value", "agent", "version"} to log
#    c. read(key): return self._store[key]["value"] if key exists, else None
#    d. keys() -> list: return list(self._store.keys())
# 3. Create bb = Blackboard(); write 3 keys from different agents
# 4. Read them back; print all keys and values
#
# Hint:
#   class Blackboard:
#       def __init__(self): self._store = {}; self._lock = threading.Lock(); self.event_log = []
#       def write(self, key, value, agent_id="system"):
#           with self._lock:
#               old_v = self._store[key]["version"] if key in self._store else 0
#               self._store[key] = {"value": value, "version": old_v+1,
#                                   "agent_id": agent_id, "timestamp": time.time()}
#               self.event_log.append({"op":"write","key":key,"value":value,
#                                      "agent":agent_id,"version":old_v+1})
#       def read(self, key): return self._store[key]["value"] if key in self._store else None

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Define compare_and_swap(bb, key, expected_version, new_value) -> bool:
#    a. Acquire bb._lock
#    b. If key not in bb._store: if expected_version == 0 write and return True; else False
#    c. If bb._store[key]["version"] != expected_version: return False (stale)
#    d. Update value and version; return True
# 2. Demonstrate optimistic concurrency conflict:
#    a. Agent A reads version N; Agent B reads version N
#    b. Agent A CAS-updates successfully (version match)
#    c. Agent B CAS fails (stale version after A's update) - print conflict warning
# 3. Print version after each successful write
#
# Hint:
#   def compare_and_swap(bb, key, expected_version, new_value):
#       with bb._lock:
#           entry = bb._store.get(key)
#           if entry is None and expected_version == 0:
#               bb._store[key] = {"value": new_value, "version": 1, ...}; return True
#           if entry["version"] != expected_version: return False
#           bb._store[key]["value"] = new_value; bb._store[key]["version"] += 1; return True

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Create bb = Blackboard(); write initial state: order_id, items, status="pending", approved=False
# 2. Simulate 5 OrderFlow agents in sequence:
#    a. validation_agent: read items, write validation_result={"valid": True}
#    b. inventory_agent: read items, write inventory_result={"all_available": True}
#    c. pricing_agent: read items, compute total, write pricing_result={"total": ...}
#    d. approval_agent: read pricing+inventory results, write approved=True
#    e. fulfilment_agent: read approved, write status="confirmed"
# 3. Print final blackboard state (all keys + values)
# 4. Print event_log summary: N writes by M agents
#
# Hint:
#   bb.write("order_id", "ORD-001")
#   bb.write("items", [{"name": "Margherita", "qty": 2, "price": 13.99}])
#   bb.write("status", "pending")
#   bb.write("validation_result", {"valid": True}, agent_id="validation_agent")
#   # ... continue for each agent ...
#   print({k: bb.read(k) for k in bb.keys()})

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Define get_event_log(bb) -> list: return a copy of bb.event_log
# 2. Define reconstruct_state(bb, up_to_n=None) -> dict:
#    Replay bb.event_log[:up_to_n]; for each "write" entry: state[key] = value
#    Return reconstructed state dict
# 3. Print full event log (op, key, agent, version per entry)
# 4. Call reconstruct_state(bb, up_to_n=3) to show state after first 3 writes
#
# Hint:
#   def reconstruct_state(bb, up_to_n=None):
#       state = {}
#       for entry in bb.event_log[:up_to_n]:
#           if entry["op"] == "write":
#               state[entry["key"]] = entry["value"]
#       return state
#   print("State after 3 writes:", reconstruct_state(bb, up_to_n=3))

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Create bb2 = Blackboard()
# 2. Define agent_worker(name, key, value, delay=0.0):
#    time.sleep(delay); bb2.write(key, value, agent_id=name)
# 3. Create 4 threads with staggered delays (0.0, 0.01, 0.02, 0.03):
#    inventory_agent, pricing_agent, validation_agent, notification_agent
# 4. Start and join all threads
# 5. Print "Concurrent writes complete" + final state + total log entries
#
# Hint:
#   threads = [
#       threading.Thread(target=agent_worker, args=("inventory_agent",   "stock",    42,    0.0)),
#       threading.Thread(target=agent_worker, args=("pricing_agent",     "total",    82.45, 0.01)),
#       threading.Thread(target=agent_worker, args=("validation_agent",  "valid",    True,  0.02)),
#       threading.Thread(target=agent_worker, args=("notification_agent","notified", True,  0.03)),
#   ]
#   for t in threads: t.start()
#   for t in threads: t.join()

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Create class TTLBlackboard (subclass Blackboard) that adds ttl_s param to write():
#    entry["expires_at"] = time.time() + ttl_s if ttl_s else None
# 2. Override read() to check expiry: if expires_at and time.time() > expires_at:
#    delete from _store; return None
# 3. Create bb3 = TTLBlackboard()
# 4. Write "quote_cache" with ttl_s=0.1; write "order_id" with no TTL
# 5. Read both immediately (both present); time.sleep(0.15)
# 6. Read again: "quote_cache" returns None (expired), "order_id" still present
#
# Hint:
#   class TTLBlackboard(Blackboard):
#       def write(self, key, value, agent_id="system", ttl_s=None):
#           with self._lock:
#               expires_at = time.time() + ttl_s if ttl_s else None
#               old_v = self._store[key]["version"] if key in self._store else 0
#               self._store[key] = {"value": value, "expires_at": expires_at,
#                                   "version": old_v+1, "agent_id": agent_id}
#       def read(self, key):
#           entry = self._store.get(key)
#           if entry and entry.get("expires_at") and time.time() > entry["expires_at"]:
#               del self._store[key]; return None
#           return entry["value"] if entry else None

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>